# DLinear-t on GIFT-Eval

Contact: sereiwathnaros@chungbuk.ac.kr (Big Data CBNU).

Model: a DLinear conditional-mean backbone (per-window instance normalisation) plus a trailing-variance MLP,
combined in a one-step Student-t output with the tail fixed at nu = 5.001 (no diffusion, no autoregression;
one forward pass gives all horizon-wise parameters and forecast samples are independent draws).
Protocol: one model per benchmark configuration, trained from scratch on the benchmark's validation split
(train + one validation horizon, as the benchmark's deep-learning baselines), univariate (multivariate
datasets are split into their variates), 100 samples per forecast, evaluated with gluonts `evaluate_model`
using the leaderboard's settings. The last H observations of each training series are held out for early
stopping and never appear as a training target. No test data is used for training or model selection.

Requirements: the gift-eval package (pip install -e .), gluonts >= 0.15, torch, numpy, pandas.

This notebook is the replication code for the `DLinear-t` leaderboard entry (`results/DLinear-t`). It is the cell-by-cell form of the script that produced the submitted numbers; running the last cell with `RUN_ALL = True` regenerates `all_results.csv` for all 97 configurations (about 45 GPU-minutes on one GPU; finished rows are skipped on restart).

Set `GIFT_EVAL` to the benchmark data directory before running (e.g. in a `.env` file, as the other notebooks do).

In [ ]:
from __future__ import annotations
import argparse, csv, json, math, os, sys, time, traceback
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F

from dotenv import load_dotenv
load_dotenv()
from gluonts.model import evaluate_model  # noqa: E402
from gluonts.model.forecast import SampleForecast  # noqa: E402
from gluonts.model.predictor import Predictor  # noqa: E402
from gluonts.time_feature import get_seasonality  # noqa: E402
from gluonts.ev.metrics import MSE, MAE, MASE, MAPE, SMAPE, MSIS, RMSE, NRMSE, ND, MeanWeightedSumQuantileLoss  # noqa: E402
from gift_eval.data import Dataset  # noqa: E402

MODEL_NAME = "DLinear-t"
CLIP = 20.0     # standardised values are winsorised to +-CLIP (cloud/traffic series contain extreme spikes)
NU0 = 5.0       # Student-t tail: nu = 2 + softplus(softplus^-1(NU0 - 2)) + 1e-3 = 5.001, kept fixed


## model

In [ ]:
# ----------------------------------------------------------------------------------------------- model
# The two sub-networks below implement published designs from their papers' descriptions:
#   mean      - the decomposition-linear forecaster of Zeng, Chen, Zhang & Xu, "Are Transformers Effective for Time
#               Series Forecasting?", AAAI 2023 (moving-average trend / remainder split, one linear map per part);
#   variance  - the rolling-variance MLP estimator of Ye, Xu & Gui, "Non-stationary Diffusion for Probabilistic
#               Time Series Forecasting", ICML 2025 (window variance of the context -> MLP -> softplus).
# All tensors are univariate: contexts are (batch, L), outputs (batch, H).
DECOMP_KERNEL = 25   # centred moving-average window for the trend
VAR_HIDDEN = 512     # width of the variance MLP


def moving_mean(x, k):  # (B, L) -> (B, L): centred mean over k steps, edges extended with the boundary value
    pad = (k - 1) // 2
    return F.avg_pool1d(F.pad(x[:, None, :], (pad, pad), mode="replicate"), k, stride=1)[:, 0, :]


def rolling_variance(x, w):  # (B, L) -> (B, L - w + 1, ): population variance of every length-w window of x
    return x.unfold(1, w, 1).var(dim=-1, unbiased=False)


class MeanNet(nn.Module):
    """Conditional mean: instance-normalised context -> trend/remainder split -> two linear maps -> de-normalised."""
    def __init__(self, L, H):
        super().__init__()
        self.remainder_map = nn.Linear(L, H)
        self.trend_map = nn.Linear(L, H)

    def forward(self, ctx):  # (B, L) -> (B, H)
        loc = ctx.mean(1, keepdim=True).detach()
        z = ctx - loc
        scale = torch.sqrt(torch.var(z, dim=1, keepdim=True, unbiased=False) + 1e-5).detach()  # statistics carry no gradient
        z = z / scale
        trend = moving_mean(z, DECOMP_KERNEL)
        out = self.remainder_map(z - trend) + self.trend_map(trend)
        return out * scale + loc


class VarianceNet(nn.Module):
    """Conditional variance: rolling variances of the context (windows that end before each step) -> MLP -> softplus."""
    def __init__(self, L, H, w):
        super().__init__()
        self.w = w
        self.fc1 = nn.Linear(L - w, VAR_HIDDEN)
        self.fc2 = nn.Linear(VAR_HIDDEN, VAR_HIDDEN)
        self.out = nn.Linear(VAR_HIDDEN, H)

    def forward(self, ctx):  # (B, L) -> (B, H)
        v = rolling_variance(ctx, self.w)[:, 1:] + 1e-7  # drop the first window: entry t summarises steps < t
        return F.softplus(self.out(F.relu(self.fc2(F.relu(self.fc1(v))))))


class DLinearT(nn.Module):
    """Mean network + variance network + fixed-tail Student-t one-step head, univariate."""
    def __init__(self, L, H, init_df=NU0):
        super().__init__()
        self.L, self.H = L, H
        self.f = MeanNet(L, H)
        self.g = VarianceNet(L, H, max(2, min(24, L // 4)))
        self.register_buffer("raw_df", torch.tensor(math.log(math.exp(max(init_df - 2.0, 1e-3)) - 1.0)))  # fixed, not a parameter

    @property
    def nu(self):
        return 2.0 + F.softplus(self.raw_df) + 1e-3

    def backbone(self, x):  # x: (B, L) -> mu (B, H), variance g (B, H)
        return self.f(x), self.g(x).clamp(min=1e-4)

    def scale(self, g):
        return torch.sqrt(g * (self.nu - 2.0) / self.nu)  # Student-t scale with variance g

    def nll(self, x, y, mask):
        mu, g = self.backbone(x)
        if not (torch.isfinite(mu).all() and torch.isfinite(g).all()):
            return torch.tensor(float("nan"), device=x.device)
        lp = torch.distributions.StudentT(df=self.nu, loc=mu, scale=self.scale(g), validate_args=False).log_prob(y)
        m = mask.float()
        return -(lp * m).sum() / m.sum().clamp(min=1.0)

    @torch.no_grad()
    def sample(self, x, S):  # -> (B, S, H)
        mu, g = self.backbone(x); sc = self.scale(g)
        z = torch.distributions.StudentT(df=self.nu, loc=torch.zeros_like(mu), scale=torch.ones_like(mu), validate_args=False).sample((S,))
        return (mu[None] + sc[None] * z).permute(1, 0, 2)


## data windows

In [ ]:
# ----------------------------------------------------------------------------------------------- data windows
def impute(x):
    """Forward/backward fill NaNs; all-NaN series become zeros."""
    if not np.isnan(x).any():
        return x.astype(np.float32)
    s = pd.Series(x.astype(np.float64)).ffill().bfill()
    return np.nan_to_num(s.to_numpy(), nan=0.0).astype(np.float32)


def series_stats(y):
    v = y[np.isfinite(y)]
    if v.size == 0:
        return 0.0, 1.0
    mu, sd = float(v.mean()), float(v.std())
    return mu, (sd if sd > 1e-8 else 1.0)


def left_pad(x, L):
    if len(x) >= L:
        return x[-L:]
    return np.concatenate([np.full(L - len(x), x[0] if len(x) else 0.0, dtype=x.dtype), x])


class SeriesStore:
    """Standardised, imputed series with NaN masks, ready for window sampling."""
    def __init__(self, entries, L, H):
        self.L, self.H = L, H
        self.series, self.masks = [], []
        for e in entries:
            y = np.asarray(e["target"], dtype=np.float32)
            if y.ndim > 1:
                raise ValueError("expected univariate targets; use to_univariate=True")
            mu, sd = series_stats(y)
            self.masks.append(np.isfinite(y)); self.series.append(np.clip((impute(y) - mu) / sd, -CLIP, CLIP))
        self.lengths = np.array([len(s) for s in self.series])
        # sampling weight: number of admissible windows per series (>= 1 so short series still appear)
        self.weights = np.maximum(self.lengths - 2 * H, 1).astype(np.float64); self.weights /= self.weights.sum()

    def window(self, i, t):
        """Context ending at t (exclusive) and horizon starting at t."""
        s, m = self.series[i], self.masks[i]
        x = left_pad(s[:t], self.L) if t > 0 else np.full(self.L, s[0] if len(s) else 0.0, np.float32)
        yy, mm = s[t:t + self.H], m[t:t + self.H]
        y = np.zeros(self.H, np.float32); msk = np.zeros(self.H, bool); y[:len(yy)] = yy; msk[:len(mm)] = mm
        return x, y, msk

    def sample_batch(self, B, rng):
        """Random windows whose horizon ends before the last H observations (those are the validation target)."""
        idx = rng.choice(len(self.series), size=B, p=self.weights)
        X = np.zeros((B, self.L), np.float32); Y = np.zeros((B, self.H), np.float32); M = np.zeros((B, self.H), bool)
        for b, i in enumerate(idx):
            T = self.lengths[i]; hi = T - 2 * self.H
            if hi <= 0:            # T <= 2H: no disjoint window exists; fall back to the in-sample range for this series
                hi = T - self.H
            t = 0 if hi <= 0 else int(rng.integers(1, hi + 1))
            X[b], Y[b], M[b] = self.window(i, t)
        return torch.from_numpy(X), torch.from_numpy(Y), torch.from_numpy(M)

    def holdout_batches(self, B, max_series=2000):
        """Deterministic validation windows: the last H observations of each series as the target."""
        n = min(len(self.series), max_series); order = np.linspace(0, len(self.series) - 1, n).astype(int)
        for k in range(0, n, B):
            sel = order[k:k + B]
            X = np.zeros((len(sel), self.L), np.float32); Y = np.zeros((len(sel), self.H), np.float32); M = np.zeros((len(sel), self.H), bool)
            for b, i in enumerate(sel):
                X[b], Y[b], M[b] = self.window(i, max(self.lengths[i] - self.H, 1))
            yield torch.from_numpy(X), torch.from_numpy(Y), torch.from_numpy(M)


## training

In [ ]:
# ----------------------------------------------------------------------------------------------- training
def context_length(H, min_len, cap=720):
    L = int(min(max(2 * H, 32), cap))
    return max(16, min(L, max(min_len - 1, 16)))  # never longer than the shortest series minus one step


def train(train_entries, L, H, steps, batch, patience=8, lr=1e-3, eval_every=100, device="cuda:0", seed=1, log=print, warmup=100):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    warmup = min(warmup, max(1, steps // 10)); eval_every = max(1, min(eval_every, steps))
    store = SeriesStore(train_entries, L, H)
    model = DLinearT(L, H).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    def lr_at(step):  # linear warm-up then cosine decay to 10% of lr
        if step <= warmup: return lr * step / max(warmup, 1)
        p = (step - warmup) / max(steps - warmup, 1); return lr * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * p)))
    best, best_state, bad, t0, bad_batches = math.inf, None, 0, time.time(), 0
    val_batch = 256 if L <= 128 else (64 if L <= 384 else 16)
    for step in range(1, steps + 1):
        model.train()
        for gq in opt.param_groups: gq["lr"] = lr_at(step)
        X, Y, M = store.sample_batch(batch, rng)
        loss = model.nll(X.to(device), Y.to(device), M.to(device))
        if not torch.isfinite(loss):
            bad_batches += 1
            if bad_batches >= 20:
                log(f"  {bad_batches} consecutive non-finite batches at step {step}; stopping at best checkpoint"); break
            continue
        bad_batches = 0
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if step % eval_every == 0 or step == steps:
            model.eval(); tot, n = 0.0, 0
            with torch.no_grad():
                for X, Y, M in store.holdout_batches(val_batch):
                    l = model.nll(X.to(device), Y.to(device), M.to(device)).item()
                    if math.isfinite(l): tot += l * len(X); n += len(X)
            val = tot / n if n else math.inf
            improved = val < best - 1e-4
            if improved:
                best, bad = val, 0; best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
            log(f"  step {step:5d} train_nll={loss.item():.4f} val_nll={val:.4f} {'*' if improved else ''} [{time.time()-t0:.0f}s]")
            if bad >= patience:
                log("  early stopping"); break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model.eval(), {"best_val_nll": (best if math.isfinite(best) else None), "steps": step, "target_steps": steps, "seconds": time.time() - t0,
                          "n_series": len(store.series), "n_points": int(store.lengths.sum()), "L": L, "H": H, "nu": float(model.nu),
                          "n_params": sum(p.numel() for p in model.parameters()), "seed": seed}


## predictor

In [ ]:
# ----------------------------------------------------------------------------------------------- predictor
class DLinearTPredictor(Predictor):
    def __init__(self, model, L, H, device, num_samples=100, batch_size=256):
        super().__init__(prediction_length=H)
        self.model, self.L, self.H, self.device, self.num_samples, self.batch_size = model.eval(), L, H, device, num_samples, batch_size
        self.n_nonfinite = 0  # forecast samples replaced by the series mean (never happened in the submitted run)

    def predict(self, dataset, **kwargs):
        entries = list(dataset); B = self.batch_size
        for k in range(0, len(entries), B):
            chunk = entries[k:k + B]; X, stats, meta = [], [], []
            for e in chunk:
                y = np.asarray(e["target"], dtype=np.float32)
                mu, sd = series_stats(y); s = np.clip((impute(y) - mu) / sd, -CLIP, CLIP)
                X.append(left_pad(s, self.L) if len(s) else np.zeros(self.L, np.float32)); stats.append((mu, sd))
                meta.append((e["start"] + len(y), e.get("item_id")))
            x = torch.from_numpy(np.stack(X)).to(self.device)
            with torch.no_grad():
                smp = self.model.sample(x, self.num_samples).cpu().numpy()  # (b, S, H)
            for b, ((mu, sd), (start, item_id)) in enumerate(zip(stats, meta)):
                samples = smp[b] * sd + mu; bad = ~np.isfinite(samples)
                if bad.any():
                    self.n_nonfinite += int(bad.sum()); samples = np.where(bad, mu, samples)
                yield SampleForecast(samples=samples, start_date=start, item_id=item_id)


## benchmark loop

In [ ]:
# ----------------------------------------------------------------------------------------------- benchmark loop
METRICS = [MSE(forecast_type="mean"), MSE(forecast_type=0.5), MAE(), MASE(), MAPE(), SMAPE(), MSIS(), RMSE(), NRMSE(), ND(),
           MeanWeightedSumQuantileLoss(quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])]
COLS = ["dataset", "model", "eval_metrics/MSE[mean]", "eval_metrics/MSE[0.5]", "eval_metrics/MAE[0.5]", "eval_metrics/MASE[0.5]", "eval_metrics/MAPE[0.5]",
        "eval_metrics/sMAPE[0.5]", "eval_metrics/MSIS", "eval_metrics/RMSE[mean]", "eval_metrics/NRMSE[mean]", "eval_metrics/ND[0.5]",
        "eval_metrics/mean_weighted_sum_quantile_loss", "domain", "num_variates"]
RES_KEYS = ["MSE[mean]", "MSE[0.5]", "MAE[0.5]", "MASE[0.5]", "MAPE[0.5]", "sMAPE[0.5]", "MSIS", "RMSE[mean]", "NRMSE[mean]", "ND[0.5]", "mean_weighted_sum_quantile_loss"]
SHORT = ["m4_yearly", "m4_quarterly", "m4_monthly", "m4_weekly", "m4_daily", "m4_hourly", "electricity/15T", "electricity/H", "electricity/D", "electricity/W",
         "solar/10T", "solar/H", "solar/D", "solar/W", "hospital", "covid_deaths", "us_births/D", "us_births/M", "us_births/W", "saugeenday/D", "saugeenday/M",
         "saugeenday/W", "temperature_rain_with_missing", "kdd_cup_2018_with_missing/H", "kdd_cup_2018_with_missing/D", "car_parts_with_missing", "restaurant",
         "hierarchical_sales/D", "hierarchical_sales/W", "LOOP_SEATTLE/5T", "LOOP_SEATTLE/H", "LOOP_SEATTLE/D", "SZ_TAXI/15T", "SZ_TAXI/H", "M_DENSE/H", "M_DENSE/D",
         "ett1/15T", "ett1/H", "ett1/D", "ett1/W", "ett2/15T", "ett2/H", "ett2/D", "ett2/W", "jena_weather/10T", "jena_weather/H", "jena_weather/D",
         "bitbrains_fast_storage/5T", "bitbrains_fast_storage/H", "bitbrains_rnd/5T", "bitbrains_rnd/H", "bizitobs_application", "bizitobs_service", "bizitobs_l2c/5T", "bizitobs_l2c/H"]
MED_LONG = ["electricity/15T", "electricity/H", "solar/10T", "solar/H", "kdd_cup_2018_with_missing/H", "LOOP_SEATTLE/5T", "LOOP_SEATTLE/H", "SZ_TAXI/15T", "M_DENSE/H",
            "ett1/15T", "ett1/H", "ett2/15T", "ett2/H", "jena_weather/10T", "jena_weather/H", "bitbrains_fast_storage/5T", "bitbrains_rnd/5T", "bizitobs_application",
            "bizitobs_service", "bizitobs_l2c/5T", "bizitobs_l2c/H"]  # the 21 datasets that also have medium and long terms (as in the benchmark notebooks)
PRETTY = {"saugeenday": "saugeen", "temperature_rain_with_missing": "temperature_rain", "kdd_cup_2018_with_missing": "kdd_cup_2018", "car_parts_with_missing": "car_parts"}


def all_configs():
    return [(n, "short") for n in SHORT] + [(n, t) for n in MED_LONG for t in ("medium", "long")]


def row_name(name, term, props):  # leaderboard row name, exactly as the benchmark notebooks build it
    key = PRETTY.get(name.split("/")[0].lower(), name.split("/")[0].lower())
    freq = name.split("/")[1] if "/" in name else props[key]["frequency"]
    return f"{key}/{freq}/{term}", key


## Run

In [ ]:
RUN_ALL = False                         # True: all 97 configurations; False: the example configurations below
CONFIGS = ["m4_weekly/W/short", "ett1/H/short"]
STEPS = {"short": 3000, "medium": 5000, "long": 6000}
SEED, PATIENCE, NUM_SAMPLES, CONTEXT_CAP = 1, 8, 100, 720

props = json.load(open("dataset_properties.json"))
out = Path("../results") / MODEL_NAME; out.mkdir(parents=True, exist_ok=True)
csv_path = out / "all_results.csv"; log_path = out / "train_log.jsonl"
done = {r["dataset"] for r in csv.DictReader(open(csv_path))} if csv_path.is_file() else set()
if not csv_path.is_file():
    with csv_path.open("w", newline="") as f: csv.writer(f).writerow(COLS)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
for name, term in all_configs():
    row, key = row_name(name, term, props)
    if not RUN_ALL and row not in CONFIGS: continue
    if row in done: print("skip (done):", row); continue
    t0 = time.time()
    probe = Dataset(name=name, term=term, to_univariate=False)
    ds = Dataset(name=name, term=term, to_univariate=(probe.target_dim > 1))
    H = ds.prediction_length; season = get_seasonality(ds.freq)
    train_entries = list(ds.validation_dataset)      # train split + one validation horizon
    L = context_length(H, min(len(e["target"]) for e in train_entries), CONTEXT_CAP)
    batch = 64 if H <= 96 else (32 if H <= 360 else 16)
    print(f"== {row}: series={len(train_entries)} H={H} L={L} freq={ds.freq} windows={ds.windows} steps={STEPS[term]}", flush=True)
    model, info = train(train_entries, L, H, steps=STEPS[term], batch=batch, patience=PATIENCE, device=device, seed=SEED, log=lambda m: print(m, flush=True))
    pred = DLinearTPredictor(model, L, H, device, num_samples=NUM_SAMPLES, batch_size=256 if L <= 128 else (64 if L <= 384 else 16))
    res = evaluate_model(pred, test_data=ds.test_data, metrics=METRICS, batch_size=512, axis=None, mask_invalid_label=True, allow_nan_forecast=False, seasonality=season)
    out_row = [row, MODEL_NAME] + [float(res[k][0]) for k in RES_KEYS] + [props[key]["domain"], props[key]["num_variates"]]
    with csv_path.open("a", newline="") as f: csv.writer(f).writerow(out_row)
    info.update({"config": row, "batch": batch, "nonfinite_samples_replaced": pred.n_nonfinite, "seconds_total": time.time() - t0, "MASE": out_row[5], "CRPS": out_row[12]})
    with log_path.open("a") as f: f.write(json.dumps(info) + "\n")
    print(f"   done: MASE={out_row[5]:.4f} CRPS={out_row[12]:.4f} [{time.time()-t0:.0f}s]", flush=True)
    model = pred = None; torch.cuda.empty_cache()
